In [3]:
import numpy as np
import pandas as pd

In [4]:
data=pd.read_csv(r"movies.csv")
data.head()

,text,sentiment
0,"Now, I won't deny that when I purchased this o...",neg
1,"The saddest thing about this ""tribute"" is that...",neg
2,Last night I decided to watch the prequel or s...,neg
3,I have to admit that i liked the first half of...,neg
4,I was not impressed about this film especially...,neg


In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       25000 non-null  object
 1   sentiment  25000 non-null  object
dtypes: object(2)
memory usage: 390.8+ KB


In [6]:
data.describe()

,text,sentiment
count,25000,25000
unique,24904,2
top,This show comes up with interesting locations ...,neg
freq,3,12500


In [7]:
data.isna().sum()

text         0
sentiment    0
dtype: int64

In [8]:
data.drop_duplicates(inplace=True)

In [9]:
data.shape

(24904, 2)

In [10]:
data['sentiment']=np.where(data['sentiment']=='neg',0,1)

In [11]:
data['sentiment'].value_counts()

sentiment
1    12472
0    12432
Name: count, dtype: int64

In [12]:
import re
def remove_htmltages(text):
    clean=re.sub(r'<.*?>', '',text)
    return clean
data['text'] = data['text'].apply(remove_htmltages)


In [13]:
def change_str(text):
    return text.lower()
data['text']=data['text'].apply(change_str)
data.head()

,text,sentiment
0,"now, i won't deny that when i purchased this o...",0
1,"the saddest thing about this ""tribute"" is that...",0
2,last night i decided to watch the prequel or s...,0
3,i have to admit that i liked the first half of...,0
4,i was not impressed about this film especially...,0


In [14]:
def remove_special_chars(text):
    if isinstance(text, str):
        return re.sub(r'[^a-zA-Z\s]', '', text)
    return text
def remove_extra_spaces(text):
    if isinstance(text, str):
        return re.sub(r'\s+', ' ', text).strip()
    return text
data['text']=data['text'].apply(remove_special_chars)
data['text']=data['text'].apply(remove_extra_spaces)

In [15]:
data.head()

,text,sentiment
0,now i wont deny that when i purchased this off...,0
1,the saddest thing about this tribute is that a...,0
2,last night i decided to watch the prequel or s...,0
3,i have to admit that i liked the first half of...,0
4,i was not impressed about this film especially...,0


In [16]:
import nltk
from nltk.corpus import stopwords

In [17]:
len(stopwords.words('english'))

198

In [18]:

stop_words = set(stopwords.words('english'))  

def remove_stopwords(text):
    if isinstance(text, str):
        x = []
        for i in text.split():
            if i not in stop_words:
                x.append(i)
        return " ".join(x)
    return text

data['text'] = data['text'].apply(remove_stopwords)

In [19]:
data.head()

,text,sentiment
0,wont deny purchased ebay high expectations inc...,0
1,saddest thing tribute almost singers including...,0
2,last night decided watch prequel shall say cal...,0
3,admit liked first half sleepers looked good ac...,0
4,impressed film especially fact went cinema fam...,0


In [20]:
from nltk.stem import PorterStemmer
ps=PorterStemmer()

In [21]:
def apply_stemming(text):
    if isinstance(text, str):
        return " ".join(ps.stem(word) for word in text.split())
    return text

In [22]:
data['text'].apply(apply_stemming)

0        wont deni purchas ebay high expect incred outo...
1        saddest thing tribut almost singer includ othe...
2        last night decid watch prequel shall say call ...
3        admit like first half sleeper look good act ev...
4        impress film especi fact went cinema famili go...
                               ...                        
24995    film fun person like good campi featur film ev...
24996    see film feel like know littl bit usa david ly...
24997    first deserv star due act would give better su...
24998    like film rambl littl plot exposit spice kinki...
24999    interest sheet cardboard dispens period piec l...
Name: text, Length: 24904, dtype: object

In [23]:
from sklearn.feature_extraction.text import CountVectorizer

cv=CountVectorizer(max_features=1000)
X = cv.fit_transform(data['text'])

In [24]:
X=X.toarray()

In [25]:
y=data.iloc[:,1]

In [26]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.3,random_state=42)

In [28]:
from sklearn.naive_bayes import GaussianNB,MultinomialNB,BernoulliNB

In [29]:
clf1=GaussianNB()
clf2=MultinomialNB()
clf3=BernoulliNB()

In [33]:
clf1.fit(X_train,y_train)
clf2.fit(X_train,y_train)
clf3.fit(X_train,y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"binarize binarize: float or None, default=0.0Threshold for binarizing (mapping to booleans) of sample features.If None, input is presumed to already consist of binary vectors.",0.0
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [34]:
y_pred1=clf1.predict(X_test)
y_pred2=clf2.predict(X_test)
y_pred3=clf3.predict(X_test)

In [35]:
from sklearn.metrics import accuracy_score

score1=accuracy_score(y_test,y_pred1)
score2=accuracy_score(y_test,y_pred2)
score3=accuracy_score(y_test,y_pred3)
score1,score2,score3

(0.7881423982869379, 0.8257494646680942, 0.8300321199143469)